<a href="https://colab.research.google.com/github/samerabouchakra88-spec/samer-repo-1/blob/main/first_taining%20of%20data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
df = pd.read_csv('/content/shouf_olive_oil_dataset (1).csv')
display(df.head())

,Year,Field_ID,Altitude_m,Annual_Rainfall_mm_year,Spring_Rainfall_mm_season,Summer_Max_Temp_C_avg,Frost_Days_count,Irrigation_L_per_tree,Tree_Age_Years,Tree_Density_per_hectare,Fertilizer_20_20_20_kg_per_tree,Soil_Type,Pest_Pressure_Index,Pruning_Intensity,Oil_Yield_kg_per_hectare,Olives_Produced_kg_per_hectare
0,2015,Ammatour_1_Field_1,947.0,1006.3204,223.7801,26.7,21.04,697.57,42,250,3.90,Loam,1.78,Heavy,1697.359004,6959.3041
1,2015,Ammatour_1_Field_2,974.8,1099.4627,264.4157,24.9,32.24,593.16,47,300,5.35,Loam,2.31,Light,1514.048907,6480.6437
2,2015,Ammatour_1_Field_3,847.0,1013.7025,284.0006,28.8,24.75,787.05,106,200,2.09,Clay,3.07,Heavy,1483.977796,6680.7120
3,2015,Ammatour_1_Field_4,811.0,996.6108,274.9777,30.0,13.44,772.30,89,250,4.02,Loam,5.16,Light,900.703884,4554.8449
4,2015,Ammatour_1_Field_5,851.3,1104.5220,303.5305,28.3,20.55,0.00,49,200,4.64,Sandy_Loam,5.37,Heavy,1278.534244,6021.2870


## Model Training: XGBoost Regressor

In [8]:
# Define features (X) and target (y)
X = df.drop(['Oil_Yield_kg_per_hectare', 'Olives_Produced_kg_per_hectare', 'Field_ID'], axis=1)
y = df['Oil_Yield_kg_per_hectare']

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

display(X.head())

,Year,Altitude_m,Annual_Rainfall_mm_year,Spring_Rainfall_mm_season,Summer_Max_Temp_C_avg,Frost_Days_count,Irrigation_L_per_tree,Tree_Age_Years,Tree_Density_per_hectare,Fertilizer_20_20_20_kg_per_tree,Pest_Pressure_Index,Soil_Type_Loam,Soil_Type_Sandy_Loam,Pruning_Intensity_Light,Pruning_Intensity_Moderate
0,2015,947.0,1006.3204,223.7801,26.7,21.04,697.57,42,250,3.90,1.78,True,False,False,False
1,2015,974.8,1099.4627,264.4157,24.9,32.24,593.16,47,300,5.35,2.31,True,False,True,False
2,2015,847.0,1013.7025,284.0006,28.8,24.75,787.05,106,200,2.09,3.07,False,False,False,False
3,2015,811.0,996.6108,274.9777,30.0,13.44,772.30,89,250,4.02,5.16,True,False,True,False
4,2015,851.3,1104.5220,303.5305,28.3,20.55,0.00,49,200,4.64,5.37,False,True,False,False


In [9]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Training set size: 160 samples
Testing set size: 40 samples


In [10]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Initialize and train the XGBoost Regressor model
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', # For regression tasks
    n_estimators=100,             # Number of boosting rounds
    learning_rate=0.1,            # Step size shrinkage to prevent overfitting
    max_depth=5,                  # Maximum depth of a tree
    subsample=0.8,                # Subsample ratio of the training instance
    colsample_bytree=0.8,         # Subsample ratio of columns when constructing each tree
    random_state=42               # For reproducibility
)

xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Squared Error (MSE): 46317.33
Root Mean Squared Error (RMSE): 215.21
R-squared (R2): 0.56


### Model Performance Summary
The XGBoost Regressor has been trained and evaluated. The R-squared value indicates how well the model explains the variance in the target variable (Oil Yield).